# Phase 5a: 实战演练场 (Simulation Drills) 🏋️

> **你的心声**: "思想上的巨人，行动上的矮子。" 🧠 vs 🖐️
> **解决方案**: 告别看懂了就算懂了。这里有三个不同难度的业务场景，**强迫**你自己写出代码。

---

## Level 1: 数据清洗与逻辑 (Warm-up) 🟢
**场景**: 运营给了一份 Excel 导出的原始数据，乱七八糟。你需要把它变成能用的 Tidy Data。

**任务**:
1.  创建一个 DataFrame (模拟脏数据)。
2.  把字符串类型的金额 (e.g., "$1,200.50") 转换成 float。
3.  把日期 ("2024/01/01") 转换成 datetime 对象。
4.  筛选出 "2024年1月" 且 "金额 > 500" 的大额订单。

In [1]:
import pandas as pd
import numpy as np

# 0. 造数据 (运行这个 cell 生成脏数据)
raw_data = {
    'Order_ID': ['A001', 'A002', 'A003', 'A004', 'A005'],
    'Date': ['2024/01/05', '2024/02/10', '2024/01/20', '2023/12/31', '2024/01/15'],
    'Amount': ['$1,200.00', '$50.50', '$800.00', '$20.00', 'MISSING'] # 这一列全是坑
}
df_raw = pd.DataFrame(raw_data)
print("原始数据:")
print(df_raw)
print(df_raw.describe())
# --- 你的代码写在下面 ---

原始数据:
  Order_ID        Date     Amount
0     A001  2024/01/05  $1,200.00
1     A002  2024/02/10     $50.50
2     A003  2024/01/20    $800.00
3     A004  2023/12/31     $20.00
4     A005  2024/01/15    MISSING
       Order_ID        Date     Amount
count         5           5          5
unique        5           5          5
top        A001  2024/01/05  $1,200.00
freq          1           1          1


In [ ]:
df_raw.columns = df_raw.columns.str.lower()
print(df_raw)
# 1. 处理 Amount 列 (去除 symbols, 转化为 float, 处理缺失值)
df_raw['amount'] = pd.to_numeric(df_raw['amount'].str.replace(r'[$,]|MISSING','',regex=True))
df_raw = df_raw.dropna(subset='amount')
print(df_raw)

# 2. 处理 Date 列 (转化为 datetime)
df_raw['date'] = pd.to_datetime(df_raw['date'])
print(df_raw)

# 3. 筛选 Jan 2024 且 Amount > 500
result = (
    (df_raw['date'].dt.year == 2024) & 
    (df_raw['date'].dt.month == 1) & 
    (df_raw['amount']>=500)
)
print(result)


  order_id        date     amount
0     A001  2024/01/05  $1,200.00
1     A002  2024/02/10     $50.50
2     A003  2024/01/20    $800.00
3     A004  2023/12/31     $20.00
4     A005  2024/01/15    MISSING
  order_id        date  amount
0     A001  2024/01/05  1200.0
1     A002  2024/02/10    50.5
2     A003  2024/01/20   800.0
3     A004  2023/12/31    20.0
  order_id       date  amount
0     A001 2024-01-05  1200.0
1     A002 2024-02-10    50.5
2     A003 2024-01-20   800.0
3     A004 2023-12-31    20.0
0     True
1    False
2     True
3    False
dtype: bool


## Level 2: A/B Test 效果评估 (Standard) 🟡
**场景**: 新功能上线一周了，老板问 "效果怎么样？要不要推全？"

**任务**:
1.  计算 Control 组和 Test 组的 CTR (Click-Through Rate)。
2.  计算 **Lift** (提升幅度)。
3.  (核心) 计算 **P-value**，判断提升是否显著 (Confidence Level 95%)。
    *   *提示*: 使用 `proportions_ztest`

In [4]:
# 0. 造数据
data_ab = {
    'Group': ['Control', 'Test'],
    'Impressions': [10000, 10000],  # 曝光量
    'Clicks': [980, 1050]           # 点击量
}
df_ab = pd.DataFrame(data_ab)
print(df_ab)

# --- 你的代码写在下面 ---

     Group  Impressions  Clicks
0  Control        10000     980
1     Test        10000    1050


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# 1. 计算 CTR (Clicks / Impressions)
df_ab['ctr'] = df_ab['Clicks'] / df_ab['Impressions']
# 2. 计算 Z-score 和 P-value
stat,p_value = proportions_ztest(df_ab['Clicks'],df_ab['Impressions'])
isSignificanceResult = np.where(p_value<0.05,'显著','不显著')
# 3. 打印结论 (Significant or Not?)
print(f"Z-score is {round(stat,2)},p_value is {round(p_value,4)} and SignificanceResult is {isSignificanceResult}")


Z-score is -1.64,p_value is 0.1012 and SignificanceResult is 不显著


## Level 3: 业务归因与决策 (Hard) 🔴
**场景**: 某日 DAU 突然下跌 10%，你需要快速定位原因。
**逻辑**: `DAU = New Users + Active Old Users`。如果是新用户跌了，找渠道；如果是老用户跌了，找产品 bug。

**任务**:
1.  用 Pandas 计算昨天和今天的 **Delta**。
2.  计算 **Contribution** (哪个部分对下跌贡献最大？)。
3.  输出一段给老板的汇报简讯 (Markdown)。

In [16]:
# 0. 造数据
metrics_yesterday = {'New_Users': 5000, 'Old_Users': 45000} # Total = 50000
metrics_today =     {'New_Users': 2000, 'Old_Users': 44000} # Total = 46000 (-8%)

# --- 你的代码写在下面 ---
# data = {metrics_yesterday,metrics_today}
# df = pd.DataFrame(metrics_yesterday)
# 1. 放入列表，并指定 Index (方便后面对减)
df = pd.DataFrame(
    [metrics_yesterday, metrics_today], 
    index=['Yesterday', 'Today']
)

print(df)

           New_Users  Old_Users
Yesterday       5000      45000
Today           2000      44000


In [23]:
df['dau'] = df['New_Users']+df['Old_Users']
# 1. 计算各部分的 Delta (Today - Yesterday)
delta = df.loc['Today'] - df.loc['Yesterday'] 
print(delta)
# 2. 计算归因 (Contribution % of Total Decline)
delta_percent = delta / delta.loc['dau']
print(delta_percent)
# 3. 自动生成一句话结论 (e.g., "下跌主要由新用户从 5000 跌至 2000 导致...")
print(f"dau{delta.loc['dau']},其中新用户{delta.loc['New_Users']}, 占比{delta_percent.loc['New_Users']};老用户{delta.loc['Old_Users']}, 占比{delta_percent.loc['Old_Users']};")


New_Users   -3000
Old_Users   -1000
dau         -4000
dtype: int64
New_Users    0.75
Old_Users    0.25
dau          1.00
dtype: float64
dau-4000,其中新用户-3000, 占比0.75;老用户-1000, 占比0.25;
